# FDR Testing via Yuanyue Li's Within-Library Self-Match

Reproduces the FDR benchmarking approach from:

> **Li et al., "Spectral entropy outperforms MS/MS dot product similarity for small-molecule compound identification"**
> *Nature Methods* 18, 1524–1531 (2021)

**Method (Section "FDR using MS/MS entropy similarity searches"):**
1. Take each spectrum in NIST as a query
2. Remove the query spectrum itself from the library (leave-one-out)
3. Search all remaining candidates within **10 ppm** precursor mass window
4. Compute **entropy similarity** to each candidate (0.05 Da MS2 tolerance)
5. At each similarity threshold:
   - **TP:** correct compound (same InChIKey14) retrieved above threshold
   - **FP:** wrong compound above threshold, but correct compound NOT above threshold
6. **FDR = FP / (FP + TP)**   *(Eq. 4 in paper)*
7. Stratify by **spectral entropy** levels (Fig. 5a in paper)

**Data:** `nist_protonated_sampled.msp` — 20,000 [M+H]+ spectra from NIST23

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ms_entropy as me
from bisect import bisect_left, bisect_right
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## 1. Parse MSP file

In [ ]:
MSP_PATH = '../data/reference_db/nist_protonated_sampled.msp'

def parse_msp(path):
    """Parse MSP file into list of spectrum dicts."""
    spectra = []
    current = {}
    peaks = []

    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current and peaks:
                    arr = np.array(peaks, dtype=np.float32)
                    # remove noise: <1% base peak intensity (per Yuanyue's methods)
                    base_peak = arr[:, 1].max()
                    arr = arr[arr[:, 1] >= 0.01 * base_peak]
                    if len(arr) > 0:
                        current['peaks'] = arr
                        spectra.append(current)
                current = {}
                peaks = []
                continue

            if ':' in line and not line[0].isdigit():
                key, val = line.split(':', 1)
                key = key.strip().lower()
                val = val.strip()
                if key == 'name':
                    current['name'] = val
                elif key == 'precursormz':
                    current['precursor_mz'] = float(val)
                elif key == 'inchikey':
                    current['inchikey14'] = val[:14]
                elif key == 'smiles':
                    current['smiles'] = val
                elif key == 'num peaks':
                    current['num_peaks'] = int(val)
            elif line[0].isdigit():
                parts = line.split()
                if len(parts) >= 2:
                    peaks.append([float(parts[0]), float(parts[1])])

    # last entry
    if current and peaks:
        arr = np.array(peaks, dtype=np.float32)
        base_peak = arr[:, 1].max()
        arr = arr[arr[:, 1] >= 0.01 * base_peak]
        if len(arr) > 0:
            current['peaks'] = arr
            spectra.append(current)

    return spectra

spectra = parse_msp(MSP_PATH)
print(f'Parsed {len(spectra)} spectra')
print(f'Unique compounds (InChIKey14): {len(set(s["inchikey14"] for s in spectra))}')
print(f'Example: {spectra[0]["name"]}, m/z={spectra[0]["precursor_mz"]}, '
      f'peaks={spectra[0]["peaks"].shape[0]}, ik14={spectra[0]["inchikey14"]}')

## 2. Compute spectral entropy for each spectrum

In [ ]:
for s in spectra:
    s['entropy'] = float(me.calculate_spectral_entropy(
        s['peaks'], clean_spectrum=True, min_ms2_difference_in_da=0.05
    ))

entropies = np.array([s['entropy'] for s in spectra])
print(f'Spectral entropy — min: {entropies.min():.2f}, median: {np.median(entropies):.2f}, '
      f'max: {entropies.max():.2f}')

# Distribution by entropy bin (matching paper's bins)
for lo, hi, label in [(0, 1, 'S=0–1'), (1, 2, 'S=1–2'), (2, 3, 'S=2–3'), (3, 99, 'S>3')]:
    n = sum(1 for e in entropies if lo <= e < hi)
    print(f'  {label}: {n} spectra ({100*n/len(spectra):.1f}%)')

## 3. Leave-one-out library search

For each query spectrum:
- Remove it from the library
- Find all candidates within **10 ppm** precursor mass window
- Compute entropy similarity (with precursor ion removal, 0.05 Da MS2 tolerance)
- Record the **best similarity** for correct matches (same InChIKey14) and wrong matches (different InChIKey14)

In [ ]:
def remove_precursor_ion(peaks, precursor_mz, tol_da=0.05):
    """Remove precursor ion from spectrum (per Yuanyue's methods)."""
    mask = np.abs(peaks[:, 0] - precursor_mz) > tol_da
    return peaks[mask]

# Sort spectra by precursor m/z for efficient window search
sorted_idx = np.argsort([s['precursor_mz'] for s in spectra])
spectra_sorted = [spectra[i] for i in sorted_idx]
mz_array = np.array([s['precursor_mz'] for s in spectra_sorted])

# Pre-clean spectra: remove precursor ion
for s in spectra_sorted:
    s['peaks_clean'] = remove_precursor_ion(s['peaks'], s['precursor_mz'])

print(f'Spectra sorted by precursor m/z: [{mz_array[0]:.4f}, ..., {mz_array[-1]:.4f}]')

In [ ]:
PPM_TOL = 10
MS2_TOL = 0.05

results = []  # list of dicts: {entropy, best_correct_sim, best_wrong_sim}

n = len(spectra_sorted)
for qi in range(n):
    query = spectra_sorted[qi]
    qmz = query['precursor_mz']
    qik = query['inchikey14']
    qpeaks = query['peaks_clean']

    if len(qpeaks) == 0:
        continue

    # 10 ppm window
    mz_lo = qmz * (1 - PPM_TOL / 1e6)
    mz_hi = qmz * (1 + PPM_TOL / 1e6)
    lo_idx = bisect_left(mz_array, mz_lo)
    hi_idx = bisect_right(mz_array, mz_hi)

    best_correct = -1.0
    best_wrong = -1.0

    for ci in range(lo_idx, hi_idx):
        if ci == qi:
            continue  # skip self
        cand = spectra_sorted[ci]
        cpeaks = cand['peaks_clean']
        if len(cpeaks) == 0:
            continue

        sim = me.calculate_entropy_similarity(
            qpeaks, cpeaks, ms2_tolerance_in_da=MS2_TOL, clean_spectra=True
        )

        if cand['inchikey14'] == qik:
            best_correct = max(best_correct, sim)
        else:
            best_wrong = max(best_wrong, sim)

    results.append({
        'entropy': query['entropy'],
        'best_correct_sim': best_correct,
        'best_wrong_sim': best_wrong,
        'has_correct_candidate': best_correct >= 0,
    })

    if (qi + 1) % 2000 == 0:
        print(f'  {qi+1}/{n} done...')

print(f'Finished: {len(results)} query spectra processed')
df = pd.DataFrame(results)

In [ ]:
# Quick sanity check
print(f'Spectra with a correct candidate in 10-ppm window: '
      f'{df["has_correct_candidate"].sum()} / {len(df)} '
      f'({100*df["has_correct_candidate"].mean():.1f}%)')
print(f'Spectra with NO candidate at all (isolated): '
      f'{((df["best_correct_sim"] < 0) & (df["best_wrong_sim"] < 0)).sum()}')
print()
print(df.describe())

## 4. Compute FDR at each similarity threshold

Following Eq. 4 from the paper:

$$\text{FDR} = \frac{\text{FP}}{\text{FP} + \text{TP}}$$

At threshold $t$:
- **TP**: query has `best_correct_sim >= t`
- **FP**: query has `best_wrong_sim >= t` AND `best_correct_sim < t`

In [ ]:
def compute_fdr_curve(df_subset, thresholds):
    """Compute FDR at each similarity threshold for a set of query results."""
    best_correct = df_subset['best_correct_sim'].values
    best_wrong = df_subset['best_wrong_sim'].values

    fdrs = []
    for t in thresholds:
        tp = np.sum(best_correct >= t)
        fp = np.sum((best_wrong >= t) & (best_correct < t))
        fdr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fdrs.append(fdr)
    return np.array(fdrs)

thresholds = np.linspace(0, 1, 201)

# Overall FDR curve
fdr_all = compute_fdr_curve(df, thresholds)

print(f'FDR at similarity 0.75: {fdr_all[np.argmin(np.abs(thresholds - 0.75))]:.4f}')
print(f'FDR at similarity 0.90: {fdr_all[np.argmin(np.abs(thresholds - 0.90))]:.4f}')
print(f'FDR at similarity 0.95: {fdr_all[np.argmin(np.abs(thresholds - 0.95))]:.4f}')

## 5. FDR vs Similarity Score — Overall (reproduces Fig. 5b)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, fdr_all, color='tab:blue', linewidth=2, label='Entropy similarity')
ax.axhline(y=0.10, color='gray', linestyle='--', alpha=0.5, label='10% FDR')
ax.axhline(y=0.05, color='gray', linestyle=':', alpha=0.5, label='5% FDR')
ax.set_xlabel('Similarity score', fontsize=12)
ax.set_ylabel('FDR', fontsize=12)
ax.set_title('FDR vs Entropy Similarity (Yuanyue approach, NIST23 [M+H]+)', fontsize=13)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. FDR stratified by spectral entropy (reproduces Fig. 5a)

Paper's key finding: low-entropy spectra (S < 1) have much higher FDR at the same similarity threshold. FDR depends on spectral information content.

In [ ]:
entropy_bins = [
    (0, 1, 'S = 0–1'),
    (1, 2, 'S = 1–2'),
    (2, 3, 'S = 2–3'),
    (3, 99, 'S > 3'),
]
colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:purple']

fig, ax = plt.subplots(figsize=(8, 5))

for (lo, hi, label), color in zip(entropy_bins, colors):
    mask = (df['entropy'] >= lo) & (df['entropy'] < hi)
    subset = df[mask]
    if len(subset) < 10:
        continue
    fdr_curve = compute_fdr_curve(subset, thresholds)
    ax.plot(thresholds, fdr_curve, color=color, linewidth=2,
            label=f'{label} (n={len(subset)})')

ax.axhline(y=0.10, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Similarity score', fontsize=12)
ax.set_ylabel('FDR', fontsize=12)
ax.set_title('FDR by Spectral Entropy Level (Fig. 5a reproduction)', fontsize=13)
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. ROC curve (reproduces Fig. 3 approach)

For each query where at least one candidate exists in the 10-ppm window, treat the best-matching candidate as the "call". Label = 1 if the best overall match is the correct compound.

In [ ]:
from sklearn.metrics import roc_curve, auc

# For ROC: query must have at least one candidate (correct or wrong)
has_any = (df['best_correct_sim'] >= 0) | (df['best_wrong_sim'] >= 0)
df_roc = df[has_any].copy()

# The "score" is the best similarity among ALL candidates
df_roc['best_sim'] = np.maximum(df_roc['best_correct_sim'], df_roc['best_wrong_sim'])
# Label: 1 if the best overall match is the correct compound
df_roc['label'] = (df_roc['best_correct_sim'] >= df_roc['best_wrong_sim']).astype(int)

fpr, tpr, _ = roc_curve(df_roc['label'], df_roc['best_sim'])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color='tab:blue', linewidth=2, label=f'Entropy similarity (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False positive rate', fontsize=12)
ax.set_ylabel('True positive rate', fontsize=12)
ax.set_title('ROC — Within-NIST23 Leave-One-Out Search', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Queries with candidates: {len(df_roc)}')
print(f'Correct top-1: {df_roc["label"].sum()} ({100*df_roc["label"].mean():.1f}%)')

## 8. Similarity threshold needed for target FDR levels

In [ ]:
def find_threshold_for_fdr(fdr_curve, thresholds, target_fdr):
    """Find the lowest similarity threshold that achieves <= target FDR."""
    # Scan from high threshold to low; find where FDR first exceeds target
    for i in range(len(thresholds) - 1, -1, -1):
        if fdr_curve[i] <= target_fdr:
            return thresholds[i]
    return None

print('Similarity threshold needed to achieve target FDR:')
print('=' * 65)
header = f'{"Entropy bin":<15} {"FDR<1%":>10} {"FDR<5%":>10} {"FDR<10%":>10}'
print(header)
print('-' * 65)

for (lo, hi, label) in entropy_bins:
    mask = (df['entropy'] >= lo) & (df['entropy'] < hi)
    subset = df[mask]
    if len(subset) < 10:
        continue
    fdr_curve = compute_fdr_curve(subset, thresholds)
    t1 = find_threshold_for_fdr(fdr_curve, thresholds, 0.01)
    t5 = find_threshold_for_fdr(fdr_curve, thresholds, 0.05)
    t10 = find_threshold_for_fdr(fdr_curve, thresholds, 0.10)
    print(f'{label:<15} {t1 or "N/A":>10} {t5 or "N/A":>10} {t10 or "N/A":>10}')

# Overall
t1 = find_threshold_for_fdr(fdr_all, thresholds, 0.01)
t5 = find_threshold_for_fdr(fdr_all, thresholds, 0.05)
t10 = find_threshold_for_fdr(fdr_all, thresholds, 0.10)
print('-' * 65)
print(f'{"Overall":<15} {t1 or "N/A":>10} {t5 or "N/A":>10} {t10 or "N/A":>10}')

## 9. TP / FP / TN counts at common thresholds

In [ ]:
print(f'{"Threshold":>10} {"TP":>8} {"FP":>8} {"TN":>8} {"FDR":>8} {"TPR":>8}')
print('-' * 55)

best_c = df['best_correct_sim'].values
best_w = df['best_wrong_sim'].values

for t in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    tp = np.sum(best_c >= t)
    fp = np.sum((best_w >= t) & (best_c < t))
    tn = np.sum((best_w < t) & (best_c < t))
    fdr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
    tpr = tp / np.sum(df['has_correct_candidate']) if df['has_correct_candidate'].sum() > 0 else 0.0
    print(f'{t:>10.2f} {tp:>8} {fp:>8} {tn:>8} {fdr:>8.4f} {tpr:>8.4f}')

## 10. Distribution of best-match similarities (TP vs FP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of best correct vs best wrong similarity
ax = axes[0]
mask_has_correct = df['has_correct_candidate']
ax.hist(df.loc[mask_has_correct, 'best_correct_sim'], bins=50, alpha=0.6,
        density=True, color='tab:green', label='Best correct match')
mask_has_wrong = df['best_wrong_sim'] >= 0
ax.hist(df.loc[mask_has_wrong, 'best_wrong_sim'], bins=50, alpha=0.6,
        density=True, color='tab:red', label='Best wrong match')
ax.set_xlabel('Entropy similarity', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Score distributions: correct vs wrong candidates', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Right: FDR and TPR on same plot
ax2 = axes[1]
ax2.plot(thresholds, fdr_all, color='tab:red', linewidth=2, label='FDR')

# Compute TPR curve
n_with_correct = df['has_correct_candidate'].sum()
tpr_curve = [np.sum(best_c >= t) / n_with_correct for t in thresholds]
ax2.plot(thresholds, tpr_curve, color='tab:green', linewidth=2, label='TPR')

ax2.axhline(y=0.05, color='gray', linestyle=':', alpha=0.5, label='5%')
ax2.set_xlabel('Similarity threshold', fontsize=12)
ax2.set_ylabel('Rate', fontsize=12)
ax2.set_title('FDR vs TPR tradeoff', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 11. Save results

In [ ]:
import os
out_dir = '../results/yuanyue_fdr'
os.makedirs(out_dir, exist_ok=True)

# Save per-query results
df.to_csv(f'{out_dir}/query_results.csv', index=False)

# Save FDR curves
fdr_df = pd.DataFrame({'threshold': thresholds, 'fdr_all': fdr_all})
for (lo, hi, label) in entropy_bins:
    mask = (df['entropy'] >= lo) & (df['entropy'] < hi)
    subset = df[mask]
    if len(subset) >= 10:
        col = f'fdr_{label.replace(" ", "").replace("–", "_").replace(">", "gt")}'
        fdr_df[col] = compute_fdr_curve(subset, thresholds)
fdr_df.to_csv(f'{out_dir}/fdr_curves.csv', index=False)

print(f'Results saved to {out_dir}/')
print(f'  query_results.csv: {len(df)} rows')
print(f'  fdr_curves.csv: {len(fdr_df)} threshold points')